In [ ]:
%load_ext autoreload
%autoreload 2
import sys 

import pandas as pd
import numpy as np

from astropy.io import fits
from astropy.table import Table

sys.path.append('./src')
from tl_cat_creator import fill_wasp_fits_template


# Create WASP-compliant input catalogues

In order to create WASP-compliant FITS catalogues, we <b>strongly</b> recommend setting up the user catalogues with the exact same naming convention as the downloaded WASP Catalogue Template. This ensures minimal effort for users filling in the FITS table. The only column that <b>must not</b> be filled in the user catalogue is "CNAME", as these are generated during the WASP validation.
</br>
</br>
However, it is possible to map the column names manually using a dictionary. In this case, the names in the user table should be the keys and the values should be the column names in the WASP Catalogue Template {'user_col_name': 'WASP_col_name'} (e.g. {'gaia_source_id': 'GAIA_ID'}. </br>

Columns existing in the user catalogue, but are not present in the WASP template will be ignored, unless explicitly mapped to a WASP column. </br>
Empty entries in the user catalogue are set to 'TNULL' in the resulting FITS file. </br> Make sure created FITS input catalogue file adheres to the WASP naming convention! </br>

**THIS NOTEBOOK IS ONLY FOR DEMONSTRSTION PURPOSES! </br> WHEN RUN WITHOUT CHANGING THE MAPPING DICT, ONLY A SUBSET OF COLUMNS WILL BE FILLED!** </br>
To ensure a completely filled table the mapping has to be provided for every column, OR the column names have to be changed to the exact names of the Catalogue Template FITS. For the latter no (or an empty) mapping dict has to be provided.

In [ ]:
fits_template_path = './WS2025XX-XXX_CatalogueTemplate.fits'
csv_cat_path = './WS2025XX-XXX_user_gen_cat.csv' # User generated csv

out_path = './WS2025XX-XXX_WASP_compliant_TEST.fits' # The FITS file we want to create (ready for WASP validation)

## Column name mapping
Below we check the different column names in the user table (csv_cat_path) and the WASP Catalogue Template (fits_template_path).
You will immediately see that most of the columns in the user table are the lowercase equivalent of the Cataluge Template columns. This is for demonstration purposes only and will vary in a real world scenario. Please note that we added the G, I and R mag columns (as Gmag, Imag and Rmag) to demonstrate the mapping. In addition, the WASP Template has a CNAME column which should not be provided by the user as CNAMEs are created during the validation process. The mapping is solely done on column names, the column order does not matter. An example can be seen in the user table where targprog/targsrvy and targid/targname have a switched ordering compared to the Catalogue Template.

In [ ]:
with fits.open(fits_template_path) as hdu:
    print('WASP CATALOGUE TEMPLATE COLUMNS: ')
    print(Table(hdu[1].data).columns)
print('---------------------------------------------------------------')
print('USER CATALOGUE COLUMNS: ')
print(pd.read_csv(csv_cat_path).columns.tolist())

## Setting up the mapping dictionary

In [ ]:
# This is of course just a subset of columns for demonstration purposes

mapping_dict = {'targsrvy': 'TARGSRVY',
                'targid': 'TARGID',
                'targprog': 'TARGPROG',
                'targname': 'TARGNAME',
                'progtemp': 'PROGTEMP',
                'obstemp': 'OBSTEMP',
                'Gmag': 'MAG_G',
                'Rmag': 'MAG_R',
                'Imag': 'MAG_I',
                'gaia_ra': 'GAIA_RA',
                'gaia_dec': 'GAIA_DEC'}

## Creation of the WASP compliant FITS
Now we create a new WASP compliant FITS file, to be uploaded for validation. </br> The headers from the Catalogue Template are preserved but a new Checksum is calculated when writing the FITS file. </br>
**When a column defined in the WASP Template is not found in the user table, the output file will contain that column filled with the template’s TNULL value.**

In [ ]:
fill_wasp_fits_template(csv_cat_path, fits_template_path, output_path=out_path,
                        mapping=mapping_dict, overwrite=True)

## Print a column subset to see if the mapping was successful.

In [ ]:
with fits.open(out_path) as hdu:
    # hdu.info()
    print(Table(hdu[1].data)[['TARGSRVY', 'PROGTEMP', 'OBSTEMP', 'TARGNAME',
                              'MAG_G', 'MAG_R', 'MAG_I', 'GAIA_RA', 'GAIA_DEC']])
    print(repr(hdu[0].header))